Import gw data analyzed

In [77]:
import os
import pandas as pd

# Load the csv file
league_name = 'rpk'  #'rpk', 'ifc', 'rbsc'

In [78]:
def get_df(league_name):
    # 1. Define the relative directory path
    folder_path = f"../data/2025-2026/{league_name}/"

    # 2. Load all available files (assuming up to gw36 are available right now)
    # We loop from 1 to 36 to build the full historic dataset
    file_names = [
        os.path.join(folder_path, f"gw{str(i).zfill(2)}_analyzed.csv")
        for i in range(1, 38+1)
    ]

    # Filter out files that don't exist yet (safeguard for future weeks like gw37, gw38)
    existing_files = [f for f in file_names if os.path.exists(f)]
    if not existing_files:
        raise FileNotFoundError(
            f"No analyzed CSV files found in directory: {folder_path}"
        )

    # 3. Read and combine all found CSV files into one master DataFrame
    return pd.concat([pd.read_csv(file) for file in existing_files], ignore_index=True)

In [79]:
df = get_df(league_name)

In [80]:
def get_dim_manager(league_name):
    # 1. Define the relative directory path
    folder_path = f"../data/2025-2026/{league_name}/"
    dim_manager_path = os.path.join(folder_path, "dim_managers.csv")
    return pd.read_csv(dim_manager_path)

In [81]:
# # check 38 entries for every player
# for no_of_entries in df.groupby('manager_id')['h2h_points'].count():
#     assert no_of_entries == 38

In [82]:
def get_player_history(df, manager_id, gw_start=1, gw_end=38, chronological=True):
    """Extracts and filters the gameweek history for a single manager."""
    # 1. Filter for the specific manager and gameweek range
    player_df = df[
        (df["manager_id"] == manager_id)
        & (df["gw_no"] >= gw_start)
        & (df["gw_no"] <= gw_end)
    ].copy()

    # 2. Sort by gameweek order
    sort_order = True if chronological else False
    player_df = player_df.sort_values(by="gw_no", ascending=sort_order)

    # 3. Reorder columns to make it readable as a personal timeline
    column_order = [
        "gw_no",
        "rank",
        "h2h_points",
        "points",
        "points_on_bench",
        "transfers_cost",
        "active_chip",
        "pnl",
    ]

    # Clean up the index so it looks like a fresh dataframe (0, 1, 2...)
    return player_df[column_order].reset_index(drop=True)

In [87]:
get_player_history(df, 1357701)

,gw_no,rank,h2h_points,points,points_on_bench,transfers_cost,active_chip,pnl
0,1,1,90,90,6,0,NaN,500
1,2,5,16,24,14,8,NaN,-100
2,3,5,39,39,8,0,NaN,-100
3,4,2,65,73,14,8,NaN,0
4,5,1,52,56,12,4,NaN,300
5,6,5,43,47,17,4,NaN,-100
6,7,5,37,41,22,4,NaN,-100
7,8,6,52,56,8,4,NaN,-100
8,9,1,43,47,19,4,NaN,400
9,10,6,52,56,11,4,NaN,-100


In [84]:
def summarize_league(league_name, gw_start=1, gw_end=38):
    """Loads all available CSV files for a specific league, computes performance

    statistics, and filters the final summary by the requested gameweek range.
    """
    df = get_df(league_name)

    # 4. Calculate Weekly Winners and Losers across the whole dataset
    max_ranks = df.groupby("gw_no")["rank"].transform("max")
    df["is_weekly_winner"] = df["rank"] == 1
    if league_name == "rbsc":
        df["is_weekly_loser"] = df["rank"] == max_ranks
    else:
        df["is_weekly_loser"] = df["rank"] >= max_ranks - 2

    # 5. NOW apply your custom gameweek range filter
    df_filtered = df[(df["gw_no"] >= gw_start) & (df["gw_no"] <= gw_end)].copy()

    # 6. Aggregate metrics per manager for the filtered range
    summary = (
        df_filtered.groupby("manager_id")
        .agg(
            total_h2h_points=("h2h_points", "sum"),
            total_points=("points", "sum"),
            bench_points=("points_on_bench", "sum"),
            total_pnl=("pnl", "sum"),
            weekly_wins=("is_weekly_winner", "sum"),
            weekly_losses=("is_weekly_loser", "sum"),
        )
        .reset_index()
    )

    # 7. Add Dense Rank based on H2H points within this specific window
    summary["overall_rank"] = (
        summary["total_h2h_points"]
        .rank(method="dense", ascending=False)
        .astype(int)
    )

    # 8. Add Meta info to keep track of the slice
    summary["gw_start"] = gw_start
    summary["gw_end"] = gw_end

    # 9. Dynamically load and merge Manager Names from the same folder
    dim_manager_df = get_dim_manager(league_name)
    summary = summary.merge(
        dim_manager_df[["id", "player_first_name", "player_last_name", "name"]],
        left_on="manager_id", right_on="id",
        how="left"
    )

    # 10. Clean layout order
    column_order = [
        "gw_start",
        "gw_end",
        "overall_rank",
        "manager_id",
        "name",
        "player_first_name",
        "player_last_name",
        "total_h2h_points",
        "total_points",
        "bench_points",
        "total_pnl",
        "weekly_wins",
        "weekly_losses",
    ]

    return summary[column_order].sort_values("overall_rank")

In [85]:
summary = summarize_league(league_name, 1, 36)

In [86]:
summary

,gw_start,gw_end,overall_rank,manager_id,name,player_first_name,player_last_name,total_h2h_points,total_points,bench_points,total_pnl,weekly_wins,weekly_losses
0,1,36,1,294329,Aekk72,Atthapon,Parkart,2192,2196,319,200,6,13
1,1,36,2,967075,Victory Goalkeres,sirawat,dulyavit,2124,2132,236,-500,4,17
4,1,36,3,2412148,ngnteam,Tawiwut,Charuwat,2105,2105,386,2750,13,14
3,1,36,4,1369948,Morty FC,Pattapong,Charoenchaipong,1954,1958,330,-1100,4,24
2,1,36,5,1357701,OnkaewmaneeN,Nithiz,Onkaewmanee,1767,1891,442,50,7,23
5,1,36,5,6149266,pairyn FC,Natthawat,Charoenkitmongkol,1767,1767,68,-1400,4,28


Add end of season prize to summary

In [ ]:
# add end of season prize 
def add_eos_prize(summary, prize: dict):
    summary_copy = summary.copy()
    # assign prize to rank
    summary_copy['end_of_season_prize'] = summary['rank'].map(prize).fillna(0).astype(int)
    #    assert summary_copy['end_of_season_prize'] == 0 # not applicable to ifc league because everybody paid upfront
    summary_copy['total_pnl'] = summary_copy['weekly_pnl'] + summary_copy['end_of_season_prize']
    return summary_copy

In [ ]:
# add_eos_prize(summary, {1:int(0.5*19000), 2:int(0.25*19000), 3:int(0.15*19000)}) # ifc
# add_eos_prize(summary, {1:3000, 2:2000, 3:1000, 6:-1000, 7:-2000, 8:-3000}) # rpk
add_eos_prize(summary, {1:int(0.5*19000), 2:int(0.25*19000), 3:int(0.15*19000)}) # rbsc

,points,weekly_pnl,gw_start,gw_end,rank,end_of_season_prize,total_pnl
name,,,,,,,
Tachapon Ratsameedara,2579,2250,1,38,1,3000,5250
Atthapon Parkart,2555,2550,1,38,2,2000,4550
Pattapong Charoenchaipong,2520,1800,1,38,3,1000,2800
sirawat dulyavit,2435,150,1,38,4,0,150
Tawiwut Charuwat,2378,-1000,1,38,5,0,-1000
Natthawat Charoenkitmongkol,2307,-1650,1,38,6,-1000,-2650
PHOOM T. YENBAMROONG,2299,-950,1,38,7,-2000,-2950
Nithiz Onkaewmanee,2203,-3150,1,38,8,-3000,-6150


In [81]:
summarize_weekly_results(df)

,name,weekly_wins,weekly_losses
46,Tone Na Ranong,3,0
0,Akin Suriyabhivadh,2,0
42,Sutthapa Soonthornthum,2,0
1,Arnon Porndhiti,2,0
28,Pimadej Siwapornpitak,2,0
37,Samote Viranuvatti,2,1
11,Gig ONEPOINT,2,1
21,Nick Thanapoomikul,2,0
40,Sirichai Jirapongphan,1,3
39,Sira H,1,0


visualization